# 图像编码 · 教师版 Notebook

配合 `course-design.qmd` 与 `image-encoding-v1.pptx` 使用。教师版包含素材生成、fallback 导出、接缝验证、RGB bits 演示和 BMP 复核。

## 第一格 · 路径检查

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter
from IPython.display import display
import struct, os

lesson_dir = Path.cwd()
assets_dir = lesson_dir / "assets"
if not assets_dir.is_dir() and lesson_dir.name == "notebooks" and (lesson_dir.parent / "assets").is_dir():
    os.chdir(lesson_dir.parent)
    lesson_dir = Path.cwd()
    assets_dir = lesson_dir / "assets"
assert assets_dir.is_dir(), f"请从本课目录启动 notebook。当前目录：{Path.cwd()}"
print("lesson directory:", lesson_dir)
print("assets directory:", assets_dir)

## Q1 / Q3 · 生成固定测试图

In [ ]:
# Q1/Q3 共用固定测试图：无随机数，每次结果一致
path = Path("assets/test-image.png")
path.parent.mkdir(parents=True, exist_ok=True)
w = h = 256
img = Image.new("RGB", (w, h))
px = img.load()
for y in range(h):
    for x in range(w):
        v = round(255 * x / (w - 1))
        px[x, y] = (v, v, v)
draw = ImageDraw.Draw(img)
draw.rectangle((12, 12, 72, 72), fill=(255, 0, 0))
draw.rectangle((73, 12, 133, 72), fill=(0, 255, 0))
draw.rectangle((184, 12, 244, 72), fill=(0, 0, 255))
draw.rectangle((0, 104, 128, 255), fill=(224, 224, 224))
draw.rectangle((136, 104, 255, 255), fill=(224, 224, 224))
draw.ellipse((0, 112, 128, 240), outline=(0, 0, 0), width=28)
draw.line((144, 240, 240, 144), fill=(0, 0, 0), width=28)
for x in [152, 160, 168, 176, 184, 192]:
    draw.line((x, 224, x, 244), fill=(255, 255, 255), width=2)
img.save(path)
assert img.size == (256, 256) and img.mode == "RGB"
print("created:", path)
display(img)

## Q1 · Demo A：Sampling

主课默认用 PPTX 两页证据页；此处用于备课导出和学生提出新尺寸时即时验证。

In [ ]:
# Q1 · Demo A：PPTX 中固定使用 sampling-*.png；课堂即时验证请用 show_sampling(n)
source_path = Path("assets/test-image.png")
assert source_path.is_file()
src = Image.open(source_path).convert("RGB")
sampling_outputs = {}

def make_sampling(n: int):
    sampled = src.resize((n, n), resample=Image.Resampling.NEAREST)
    enlarged = sampled.resize((512, 512), resample=Image.Resampling.NEAREST)
    assert sampled.mode == "RGB"
    assert enlarged.mode == "RGB"
    assert enlarged.size == (512, 512)
    return enlarged

for n in [256, 64, 32, 16, 8]:
    sampling_outputs[n] = make_sampling(n)
    print(f"{n} × {n}")
    display(sampling_outputs[n])

def show_sampling(n: int):
    """课堂即时验证：输入学生提出的新尺寸，例如 show_sampling(24)。"""
    assert isinstance(n, int) and n > 0
    out = make_sampling(n)
    print(f"{n} × {n}")
    display(out)
    return out

## Q2 · 生成并检查同构图反例

In [ ]:
# Q2 · 生成同场景、同 4:3 构图的图片：1600×1200 模糊 vs 800×600 清晰
from PIL import ImageFilter

def make_q2_scene(size):
    W, H = size
    im = Image.new('RGB', size, (242, 245, 248))
    p = im.load()
    for y in range(H):
        for x in range(W):
            r = 230 + int(15 * x / W)
            g = 235 + int(10 * y / H)
            b = 248 - int(25 * x / W)
            p[x, y] = (r, g, b)
    d = ImageDraw.Draw(im)
    for t in range(0, W, max(1, W//16)):
        d.line((t, 0, t, H), fill=(220, 225, 232), width=max(1, W//900))
    for t in range(0, H, max(1, H//12)):
        d.line((0, t, W, t), fill=(220, 225, 232), width=max(1, H//700))
    d.rectangle((int(W*.08), int(H*.12), int(W*.30), int(H*.34)), fill=(255, 80, 80))
    d.rectangle((int(W*.30), int(H*.12), int(W*.52), int(H*.34)), fill=(80, 210, 90))
    d.rectangle((int(W*.70), int(H*.12), int(W*.92), int(H*.34)), fill=(80, 100, 245))
    d.ellipse((int(W*.10), int(H*.48), int(W*.42), int(H*.90)), outline=(30,30,30), width=max(10, W//55))
    d.line((int(W*.58), int(H*.88), int(W*.90), int(H*.48)), fill=(30,30,30), width=max(10, W//50))
    for i in range(12):
        x = int(W*.60) + i*max(2, int(W*.012))
        d.line((x, int(H*.70), x, int(H*.85)), fill=(255,255,255), width=max(2, W//600))
    return im

high = make_q2_scene((1600, 1200)).filter(ImageFilter.GaussianBlur(radius=10))
low = make_q2_scene((800, 600))
high.save("assets/fallback/q2-high-pixels-blur.png")
low.save("assets/fallback/q2-low-pixels-sharp.png")
assert Image.open("assets/fallback/q2-high-pixels-blur.png").size == (1600, 1200)
assert Image.open("assets/fallback/q2-low-pixels-sharp.png").size == (800, 600)
print("Q2 images generated: 1600×1200 blurred, 800×600 sharp")

## Q3 · Demo B：Quantization

先看图，不先公布接缝 RGB。

In [ ]:
# Q3 · Demo B：主课只看 original / 16-color / 2-color；先看图，后揭示接缝数值
source_path = Path("assets/test-image.png")
assert source_path.is_file()
src = Image.open(source_path).convert("RGB")
targets = [256, 16, 4, 2]
main_targets = [16, 2]
outputs = {}
for n in targets:
    q = src.quantize(colors=n, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.NONE).convert("RGB")
    colors = q.getcolors(maxcolors=n + 1)
    assert colors is not None
    color_count = len(colors)
    assert q.size == src.size
    assert q.mode == "RGB"
    assert color_count <= n
    outputs[n] = q
print("original")
display(src)
for n in main_targets:
    print(f"max {n} colors")
    display(outputs[n])

## Q3 · 接缝 RGB 证据

In [ ]:
# Q3 · 接缝证据：学生观察解释之后再运行/揭示
seam_left = (72, 40)
seam_right = (73, 40)
for n in [16, 2]:
    left_rgb = outputs[n].getpixel(seam_left)
    right_rgb = outputs[n].getpixel(seam_right)
    print(f"{n:2d} colors | left={left_rgb} | right={right_rgb} | same={left_rgb == right_rgb}")

## Q6 · Demo C：一个 RGB 像素怎样写成 bits

In [ ]:
# Q6 · Demo C：RGB → bits
rgb = (255, 128, 0)
for name, value in zip(["R", "G", "B"], rgb):
    print(f"{name} {value:3d} → {value:08b}")
print("combined:", " ".join(f"{v:08b}" for v in rgb))

## 备课导出与复核 fallback

In [ ]:
# 备课导出 fallback 图片：PPTX、live demo、故障备用图使用同一批证据
fallback_dir = Path("assets/fallback")
fallback_dir.mkdir(parents=True, exist_ok=True)
for n in [256, 64, 32, 16, 8]:
    img = sampling_outputs[n]
    assert img.mode == "RGB" and img.size == (512, 512)
    img.save(fallback_dir / f"sampling-{n}.png")
quant_images = {"original": src, "16": outputs[16], "2": outputs[2]}
for label, img in quant_images.items():
    assert img.mode == "RGB" and img.size == (256, 256)
    img.save(fallback_dir / f"quant-{label}.png")
print("fallback images exported to:", fallback_dir)

In [ ]:
# 备课复核：重新打开导出的文件检查尺寸、mode 与红绿接缝
fallback_dir = Path("assets/fallback")
for n in [256, 64, 32, 16, 8]:
    with Image.open(fallback_dir / f"sampling-{n}.png") as im:
        assert im.mode == "RGB"
        assert im.size == (512, 512)
for label in ["original", "16", "2"]:
    with Image.open(fallback_dir / f"quant-{label}.png") as im:
        assert im.mode == "RGB"
        assert im.size == (256, 256)
seam_left = (72, 40)
seam_right = (73, 40)
with Image.open(fallback_dir / "quant-16.png").convert("RGB") as q16:
    assert q16.getpixel(seam_left) != q16.getpixel(seam_right)
with Image.open(fallback_dir / "quant-2.png").convert("RGB") as q2:
    assert q2.getpixel(seam_left) == q2.getpixel(seam_right)
print("fallback verification passed")

## Q7 / Q11 · BMP 文件证据

In [ ]:
# Q7/Q11 · BMP 证据复核
path = Path("assets/q7-3x2-24bit.bmp")
assert path.is_file()
data = path.read_bytes()
pixel_offset = struct.unpack_from("<I", data, 10)[0]
width = struct.unpack_from("<i", data, 18)[0]
height = struct.unpack_from("<i", data, 22)[0]
bpp = struct.unpack_from("<H", data, 28)[0]
rows = abs(height)
color_value_bytes = width * rows * bpp // 8
raw_row_bytes = width * bpp // 8
stored_row_bytes = ((width * bpp + 31) // 32) * 4
pixel_array_bytes = stored_row_bytes * rows
print("pixel-data offset :", pixel_offset, "bytes")
print("color values      :", color_value_bytes, "bytes")
print("raw row values    :", raw_row_bytes, "bytes/row")
print("stored row size   :", stored_row_bytes, "bytes/row")
print("pixel array       :", pixel_array_bytes, "bytes")
print("whole file        :", len(data), "bytes")
assert pixel_offset == 54
assert color_value_bytes == 18
assert stored_row_bytes == 12
assert pixel_array_bytes == 24
assert len(data) == 78

## 教师可选 · BMP biBitCount 检查

In [ ]:
# 教师可选：把量化结果另存为 BMP，检查磁盘文件仍是 24 bpp
out_dir = Path("demo-output/quantization-bmp")
out_dir.mkdir(parents=True, exist_ok=True)
for n, q in outputs.items():
    bmp_path = out_dir / f"max-{n}-colors.bmp"
    q.save(bmp_path, format="BMP")
    data = bmp_path.read_bytes()
    bits_per_pixel = struct.unpack_from("<H", data, 28)[0]
    print(bmp_path.name, "| bpp =", bits_per_pixel, "| file bytes =", len(data))
    assert bits_per_pixel == 24
    assert len(data) == 196662